# DataGastro CABA
## Informe ejecutivo reproducible

**Objetivo:** presentar una lectura ordenada del ecosistema gastronómico de CABA a partir de fuentes separadas y trazables.

Esta notebook no reemplaza al dashboard. Sirve como pieza narrativa para una reunión: muestra solo los indicadores defendibles, explica limitaciones y deja claro qué decisiones puede apoyar hoy el proyecto.

## 0. Cómo leer este informe

Regla metodológica central: **no se suman fuentes que miden cosas distintas**.

- **F01** mide oferta gastronómica registrada en fuente pública.
- **F02** mide habilitaciones aprobadas, no locales activos.
- **F03** mide espacios reales de ferias, mercados y FIAB; los puestos/personas no son espacios.
- **F04** releva eventos gastronómicos trazables, pero no es universo completo.
- **F05** releva programas/políticas, pero no mide impacto ni presupuesto ejecutado.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / "data" / "processed").exists() and (p / "data" / "analytics").exists():
            return p
    raise FileNotFoundError("No encuentro la raiz del proyecto. Ejecutar desde la raiz o desde notebooks/.")

ROOT = find_project_root()
DATA = ROOT / "data"
PROCESSED = DATA / "processed"
ANALYTICS = DATA / "analytics"
RAW = DATA / "raw"

print(f"Raiz del proyecto: {ROOT}")

def read_csv(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        print(f"WARN: no existe {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)

resumen = read_csv(ANALYTICS / "analytics_resumen_ejecutivo.csv")
f01 = read_csv(PROCESSED / "fact_establecimiento.csv")
f02 = read_csv(PROCESSED / "fact_habilitacion_gastronomica.csv")
f03 = read_csv(PROCESSED / "fact_espacio_feria_mercado.csv")
f04 = read_csv(PROCESSED / "fact_evento_gastronomico.csv")
f05 = read_csv(PROCESSED / "fact_programa_politica.csv")

f01_cat_barrio = read_csv(ANALYTICS / "analytics_establecimientos_por_categoria_barrio.csv")
f02_anio = read_csv(ANALYTICS / "analytics_habilitaciones_por_anio.csv")
f02_cat = read_csv(ANALYTICS / "analytics_habilitaciones_por_categoria.csv")
f02_comuna = read_csv(ANALYTICS / "analytics_habilitaciones_por_barrio.csv")
f03_tipo = read_csv(ANALYTICS / "analytics_espacios_ferias_mercados_por_tipo.csv")
mapa_op = read_csv(ANALYTICS / "analytics_mapa_oportunidades.csv")
f04_tipo = read_csv(ANALYTICS / "analytics_eventos_por_tipo.csv")
programas_catalogo = read_csv(ANALYTICS / "analytics_programas_catalogo.csv")
puente_evento_programa = read_csv(PROCESSED / "puente_evento_programa.csv")

## 1. Resumen ejecutivo

Los conteos principales se muestran separados por concepto para evitar errores de interpretación. En especial, **F02 no es un padrón de locales activos** y **F03 no cuenta puestos/personas como ferias o mercados**.

In [ ]:
metrics = {
    "Oferta registrada F01": len(f01),
    "Habilitaciones F02": len(f02),
    "Espacios F03": len(f03),
    "Eventos F04 aptos": int((f04.get("apto_dashboard", pd.Series(dtype=str)).astype(str).str.lower() == "si").sum()) if not f04.empty else 0,
    "Programas F05 aptos": int((f05.get("apto_dashboard", pd.Series(dtype=str)).astype(str).str.lower() == "si").sum()) if not f05.empty else 0,
}
metrics_df = pd.DataFrame([{"Indicador": k, "Valor": v} for k, v in metrics.items()])
metrics_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
plot_df = metrics_df.sort_values("Valor")
ax.barh(plot_df["Indicador"], plot_df["Valor"])
ax.set_title("Indicadores principales separados por fuente/concepto")
ax.set_xlabel("Cantidad")
for i, v in enumerate(plot_df["Valor"]):
    ax.text(v, i, f" {v:,.0f}".replace(",", "."), va="center")
plt.tight_layout()
plt.show()

## 2. Fuentes: qué mide cada una y qué no mide

In [ ]:
fuentes = pd.DataFrame([
    {"Fuente": "F01", "Mide": "Oferta gastronómica registrada", "No mide": "Vigencia actual ni actividad real por local", "Uso recomendado": "Mapa de oferta y distribución por barrio/categoría"},
    {"Fuente": "F02", "Mide": "Habilitaciones gastronómicas aprobadas", "No mide": "Locales activos, bajas o aperturas netas", "Uso recomendado": "Dinamismo formal por año/categoría/comuna identificada"},
    {"Fuente": "F03", "Mide": "Espacios reales de ferias, mercados y FIAB", "No mide": "Puestos/personas como espacios", "Uso recomendado": "Capa territorial de abastecimiento y espacios públicos"},
    {"Fuente": "F04", "Mide": "Eventos gastronómicos relevados y trazables", "No mide": "Universo completo de eventos", "Uso recomendado": "Contexto de activaciones y calendario institucional"},
    {"Fuente": "F05", "Mide": "Programas, políticas e instrumentos", "No mide": "Impacto, empleo, facturación o presupuesto ejecutado", "Uso recomendado": "Catálogo institucional y vínculos evento-programa"},
])
fuentes

## 3. Oferta gastronómica registrada (F01)

Esta sección usa registros de oferta gastronómica con ubicación de fuente oficial. Es útil para ver concentración territorial, pero no debe interpretarse como padrón vivo de locales activos.

In [ ]:
if not f01_cat_barrio.empty:
    top_barrios = (
        f01_cat_barrio.groupby("barrio", as_index=False)["cantidad_establecimientos"].sum()
        .sort_values("cantidad_establecimientos", ascending=False)
        .head(15)
    )
    total_f01 = top_barrios["cantidad_establecimientos"].sum()
    top3 = top_barrios.head(3)
    print("Top 3 barrios F01:")
    display(top3)

    fig, ax = plt.subplots(figsize=(9, 6))
    p = top_barrios.sort_values("cantidad_establecimientos")
    ax.barh(p["barrio"], p["cantidad_establecimientos"])
    ax.set_title("F01 · Top 15 barrios por oferta gastronómica registrada")
    ax.set_xlabel("Registros")
    plt.tight_layout()
    plt.show()
else:
    print("No se encontró analytics_establecimientos_por_categoria_barrio.csv")

In [ ]:
if not f01_cat_barrio.empty:
    top_categorias_f01 = (
        f01_cat_barrio.groupby("categoria_general", as_index=False)["cantidad_establecimientos"].sum()
        .sort_values("cantidad_establecimientos", ascending=False)
        .head(10)
    )
    fig, ax = plt.subplots(figsize=(9, 4.8))
    p = top_categorias_f01.sort_values("cantidad_establecimientos")
    ax.barh(p["categoria_general"], p["cantidad_establecimientos"])
    ax.set_title("F01 · Categorías principales de oferta registrada")
    ax.set_xlabel("Registros")
    plt.tight_layout()
    plt.show()
    display(top_categorias_f01)

## 4. Dinamismo formal: habilitaciones aprobadas (F02)

F02 es uno de los activos analíticos más fuertes del proyecto, pero debe leerse correctamente: **son habilitaciones aprobadas, no locales activos**. Además, la serie excluye recursos no comparables cuando corresponde.

In [ ]:
if not f02_anio.empty:
    anio = f02_anio.copy()
    anio["cantidad_habilitaciones"] = pd.to_numeric(anio["cantidad_habilitaciones"], errors="coerce")
    comparable = anio[anio.get("comparable_como_flujo_anual", "si").astype(str).str.lower().eq("si")].copy()
    no_comparable = anio[~anio.index.isin(comparable.index)].copy()

    print("Periodos no comparables o con advertencia metodológica:")
    display(no_comparable[[c for c in ["anio_fuente", "cantidad_habilitaciones", "nota_serie"] if c in no_comparable.columns]])

    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.bar(comparable["anio_fuente"].astype(str), comparable["cantidad_habilitaciones"])
    ax.set_title("F02 · Habilitaciones gastronómicas aprobadas por año comparable")
    ax.set_xlabel("Año")
    ax.set_ylabel("Habilitaciones")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No se encontró analytics_habilitaciones_por_anio.csv")

In [ ]:
if not f02_cat.empty:
    cat = f02_cat.copy()
    cat["cantidad_habilitaciones"] = pd.to_numeric(cat["cantidad_habilitaciones"], errors="coerce")
    top = cat.sort_values("cantidad_habilitaciones", ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    p = top.sort_values("cantidad_habilitaciones")
    ax.barh(p["categoria_gastronomica_inferida"], p["cantidad_habilitaciones"])
    ax.set_title("F02 · Habilitaciones por categoría gastronómica inferida")
    ax.set_xlabel("Habilitaciones")
    plt.tight_layout()
    plt.show()
    display(top)

## 5. Territorio: lectura por comuna

La lectura territorial de F02 debe explicitar cobertura. Hoy F02 se puede mirar por comuna cuando la fuente trae o permite identificar comuna; no debe mapearse como punto hasta contar con geocodificación validada.

In [ ]:
if not mapa_op.empty:
    cols = [c for c in [
        "comuna", "cantidad_establecimientos_f01", "cantidad_habilitaciones_f02_con_comuna",
        "cantidad_espacios_f03", "cantidad_eventos_f04_aptos", "observaciones"
    ] if c in mapa_op.columns]
    display(mapa_op[cols].sort_values("comuna"))

    plot_cols = [c for c in ["cantidad_establecimientos_f01", "cantidad_habilitaciones_f02_con_comuna", "cantidad_espacios_f03"] if c in mapa_op.columns]
    comuna_plot = mapa_op[["comuna", *plot_cols]].copy()
    for c in plot_cols:
        comuna_plot[c] = pd.to_numeric(comuna_plot[c], errors="coerce").fillna(0)
    comuna_plot = comuna_plot.sort_values("cantidad_habilitaciones_f02_con_comuna" if "cantidad_habilitaciones_f02_con_comuna" in plot_cols else plot_cols[0], ascending=False)
    display(comuna_plot.head(10))
else:
    print("No se encontró analytics_mapa_oportunidades.csv")

## 6. Espacios feriales, mercados y FIAB (F03)

F03 fue corregida a nivel de **espacio real**. Los puestos/personas se conservan, si corresponde, como insumo técnico, pero no se exponen como KPI principal ni se interpretan como ferias/mercados.

In [ ]:
if not f03_tipo.empty:
    f03_plot = f03_tipo.copy()
    f03_plot["cantidad_espacios"] = pd.to_numeric(f03_plot["cantidad_espacios"], errors="coerce")
    fig, ax = plt.subplots(figsize=(8, 4.8))
    p = f03_plot.sort_values("cantidad_espacios")
    ax.barh(p["tipo_espacio"], p["cantidad_espacios"])
    ax.set_title("F03 · Espacios reales por tipo")
    ax.set_xlabel("Espacios")
    plt.tight_layout()
    plt.show()
    display(f03_plot.sort_values("cantidad_espacios", ascending=False))
else:
    print("No se encontró analytics_espacios_ferias_mercados_por_tipo.csv")

## 7. Ecosistema público: eventos y programas (F04/F05)

F04 y F05 no deben leerse como medición de impacto. Su valor actual es ordenar el ecosistema institucional: qué eventos, programas, instrumentos y vínculos aparecen documentados.

In [ ]:
if not f04_tipo.empty:
    f04_plot = f04_tipo.copy()
    cantidad_col = "cantidad_eventos" if "cantidad_eventos" in f04_plot.columns else f04_plot.select_dtypes(include=[np.number]).columns[0]
    fig, ax = plt.subplots(figsize=(8, 4.8))
    p = f04_plot.sort_values(cantidad_col)
    ax.barh(p["tipo_evento"], p[cantidad_col])
    ax.set_title("F04 · Eventos aptos por tipo")
    ax.set_xlabel("Eventos")
    plt.tight_layout()
    plt.show()
    display(f04_plot)

if not programas_catalogo.empty:
    print("Programas/políticas aptas como catálogo:")
    display(programas_catalogo[[c for c in ["id_programa", "nombre_programa", "tipo_programa", "estado", "url_fuente"] if c in programas_catalogo.columns]])

if not puente_evento_programa.empty:
    print("Vínculos evento-programa registrados:")
    display(puente_evento_programa.head(20))

## 8. Qué puede responder hoy

- ¿Dónde se concentra la oferta gastronómica registrada?
- ¿Qué categorías predominan en F01?
- ¿Cómo evolucionan las habilitaciones gastronómicas aprobadas en años comparables?
- ¿Qué categorías explican la mayor parte de las habilitaciones F02?
- ¿Qué espacios reales de ferias, mercados y FIAB están registrados?
- ¿Qué eventos y programas aparecen documentados y trazables?

## 9. Qué todavía NO puede responder

- No dice cuántos locales gastronómicos están activos hoy.
- No mide cierres ni aperturas netas.
- No mide empleo, ventas ni impacto económico.
- No mide impacto de eventos o programas.
- No permite afirmar saturación o vacíos reales sin denominadores adicionales.
- No debe mapear F02 como puntos hasta tener geocodificación validada y auditable.

## 10. Próximos pasos recomendados

1. Cerrar validación territorial GeoJSON sin errores.
2. Consolidar dashboard V2 narrativo.
3. Usar la coropleta F02 por comuna solo con cobertura explicitada.
4. Preparar `geo_cache` y geocodificación USIG de F02 con validación muestral.
5. Integrar permisos de área gastronómica como F06.
6. Exportar este informe a HTML/PDF para enviar después de la demo.

**Criterio:** priorizar mejoras de alto impacto y bajo riesgo antes de agregar complejidad.